# Unit 6 Project
### Creating Designs by Leveraging Stable Diffusion and Gradio UI


In [ ]:
%pip install --upgrade pip
%pip install gradio 
%pip install diffusers[torch] 
%pip install triton
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu130

In [1]:
# imports
import gc
import gradio as gr
import torch
from diffusers import DiffusionPipeline, StableDiffusion3Pipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

/proj/ksims/Classwork_ml/RNNs/.venv/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


In [ ]:
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()

def generate_image (prompt, negative_prompt, guidance_scale, seed):
    # Take model and generate an image from prompt and seed
    # model_id = "stabilityai/stable-diffusion-xl-base-1.0"
    model_id = "stable-diffusion-v1-5/stable-diffusion-v1-5"
    torchtype = None
    if device == "cuda":
        torchtype = torch.float16
    pipe = DiffusionPipeline.from_pretrained(model_id, torch_dtype=torchtype, use_safetensors=True, variant="fp16", safety_checker=None)
    pipe = pipe.to(device)
    generator = torch.Generator(device).manual_seed(seed)
    pipe.to(device)
    if device == "cuda":
        pipe.enable_model_cpu_offload()
    image = pipe(prompt, negative_prompt=negative_prompt, generator=generator, height=328, width=720, guidance_scale=guidance_scale).images
    # image1 = refiner(prompt, image=image).images[0]
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()
    
    # If torch.compile was used:
    torch._dynamo.reset()
    if device == "cuda":
        torch.cuda.synchronize()
    return image[0]

In [ ]:
def_prompt = "Create a banner ad for 'Gears Ltd.' They make gears. Use blue for background color, and steel for the gears"
demo = gr.Interface(fn=generate_image , inputs=[gr.Textbox(label="Prompt", value=def_prompt),
                                             gr.Textbox(label="Negative Prompt", value="text, words, logo, watermarks, signatures"),
                                             gr.Slider(0, 20, step=0.1, label="Guidance Scale", value=7.5),
                                             gr.Number(label="Seed", value=691)],
                                     outputs=[gr.Image(label="Banner Background")],)

demo.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.




Loading weights: 100%|██████████| 196/196 [00:00<00:00, 321.12it/s]
CLIPTextModel LOAD REPORT from: /home/ksims/.cache/huggingface/hub/models--stable-diffusion-v1-5--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading pipeline components...: 100%|██████████| 6/6 [00:09<00:00,  1.59s/it]
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recomm